# ARC-AGI-3 Harness — Kaggle-Parity Local Qwen Run

This notebook imitates the thin Kaggle entrypoint: offline wheel discovery, local ARC environment loading, one centralized configuration, immutable run artifacts, trace verification, and research export. The LM remains the agent; the notebook only composes adapters.

> Ollama is a **local parity backend** here. A Kaggle kernel cannot call Ollama on your workstation. For an actual Kaggle submission, keep the harness/environment cells and replace only the model adapter with an attached-weight backend.

In [ ]:
# CENTRAL CONFIG — change experiment settings only in this cell.
from dataclasses import dataclass
from pathlib import Path
import os

HERE = Path.cwd().resolve()
REPO_ROOT = HERE if (HERE / 'pyproject.toml').exists() else HERE.parent
LOCAL_DATASET = REPO_ROOT / 'arc-prize-2026-arc-agi-3'

@dataclass(frozen=True)
class NotebookConfig:
    dataset_root: Path = Path(os.getenv('ARC_DATASET_ROOT', str(LOCAL_DATASET)))
    harness_wheels: Path = Path(os.getenv('HARNESS_WHEELS', str(REPO_ROOT / 'dist' / 'wheelhouse')))
    arc_wheels: Path = Path(os.getenv('ARC_WHEELS', str(LOCAL_DATASET / 'arc_agi_3_wheels')))
    output_root: Path = Path(os.getenv('ARC_OUTPUT_ROOT', str(REPO_ROOT / 'runs')))
    install_wheels: bool = bool(os.getenv('KAGGLE_KERNEL_RUN_TYPE'))
    game_id: str = 'ls20'
    seed: int = 0
    run_id: str = 'notebook-qwen-ls20'
    experiment_id: str = 'local-qwen-kaggle-parity'
    max_turns: int = 1  # fast smoke; raise for an exploratory episode
    checkpoint_every: int = 1
    model: str = 'qwen3.5:9b'
    ollama_url: str = 'http://127.0.0.1:11434'
    temperature: float = 0.0
    model_seed: int = 0
    num_ctx: int = 32768
    timeout_seconds: float = 300.0
    max_repairs: int = 1
    action_budget: int = 8
    model_call_budget: int = 8
    input_token_budget: int = 200_000
    output_token_budget: int = 20_000

CONFIG = NotebookConfig()
CONFIG

In [ ]:
# Kaggle/offline bootstrap. Local development uses the active editable environment.
import subprocess
import sys

if CONFIG.install_wheels:
    roots = [CONFIG.harness_wheels, CONFIG.arc_wheels]
    missing = [str(path) for path in roots if not path.is_dir()]
    if missing:
        raise FileNotFoundError(f'Missing attached wheel directories: {missing}')
    command = [sys.executable, '-m', 'pip', 'install', '--no-index']
    for root in roots:
        command.extend(['--find-links', str(root)])
    command.append('arc-agi-3[arc]')
    subprocess.run(command, check=True)
else:
    print('Local mode: using the active environment; no installation performed.')

In [ ]:
# Resolve and record the exact local model before the run.
import json
from urllib.request import urlopen

with urlopen(CONFIG.ollama_url.rstrip('/') + '/api/tags', timeout=10) as response:
    installed = json.loads(response.read()).get('models', [])
model_info = next((item for item in installed if item.get('name') == CONFIG.model), None)
if model_info is None:
    raise RuntimeError(f'{CONFIG.model!r} is not installed; run: ollama pull {CONFIG.model}')
MODEL_DIGEST = model_info['digest']
{'model': CONFIG.model, 'digest': MODEL_DIGEST, 'details': model_info.get('details', {})}

In [ ]:
# Compose the official offline environment and replaceable model adapter.
from arc_agi import Arcade
from arc_agi.base import OperationMode
from arc_agi_3.adapters.arc import ArcEnvironmentAdapter
from arc_agi_3.adapters.ollama import OllamaBackend, OllamaConfig
from arc_agi_3.adapters.structured_model import StructuredModelAdapter
from arc_agi_3.config import BudgetConfig, RunConfig
from arc_agi_3.contracts.enums import Resource
from arc_agi_3.runtime import build_runner

environment_dir = CONFIG.dataset_root / 'environment_files'
if not environment_dir.is_dir():
    raise FileNotFoundError(f'ARC environment directory not found: {environment_dir}')
arcade = Arcade(operation_mode=OperationMode.OFFLINE, environments_dir=str(environment_dir))
wrapper = arcade.make(CONFIG.game_id, seed=CONFIG.seed, include_frame_data=True)
if wrapper is None:
    raise RuntimeError(f'Could not load game {CONFIG.game_id!r}')
environment = ArcEnvironmentAdapter(wrapper)
backend = OllamaBackend(OllamaConfig(
    model=CONFIG.model, model_digest=MODEL_DIGEST, base_url=CONFIG.ollama_url,
    temperature=CONFIG.temperature, seed=CONFIG.model_seed, num_ctx=CONFIG.num_ctx,
    timeout_seconds=CONFIG.timeout_seconds,
))
model = StructuredModelAdapter(backend, max_repairs=CONFIG.max_repairs)

In [ ]:
# Build and run. Existing traces are never overwritten.
run_config = RunConfig(
    run_id=CONFIG.run_id, experiment_id=CONFIG.experiment_id, game_id=CONFIG.game_id,
    seed=CONFIG.seed, max_turns=CONFIG.max_turns, output_dir=CONFIG.output_root,
    checkpoint_every=CONFIG.checkpoint_every,
    budget=BudgetConfig(limits={
        Resource.ACTIONS: CONFIG.action_budget,
        Resource.MODEL_CALLS: CONFIG.model_call_budget,
        Resource.INPUT_TOKENS: CONFIG.input_token_budget,
        Resource.OUTPUT_TOKENS: CONFIG.output_token_budget,
    }),
)
runner = build_runner(run_config, environment, model)
result = runner.run()
result.model_dump(mode='json')

In [ ]:
# Verify agency causality and the immutable hash chain.
from arc_agi_3.audit import audit_agency_boundary
from arc_agi_3.trace.store import JsonlEventStore

events = JsonlEventStore(runner.events.path, 'verify', 'verify', 'verify').read()
violations = audit_agency_boundary(events)
assert not violations, violations
{'events': len(events), 'last_hash': events[-1].event_hash, 'agency_violations': violations}

In [ ]:
# Build disposable research artifacts without replacing the raw trace.
from arc_agi_3.research import TraceIndex, export_rows, run_row

run_dir = CONFIG.output_root / CONFIG.run_id
indexed = TraceIndex(run_dir / 'events.sqlite3').rebuild(run_dir / 'events.jsonl')
row = run_row(run_dir)
export_rows([row], run_dir / 'metrics.jsonl')
{'indexed_events': indexed, 'metrics': row, 'run_dir': str(run_dir)}